## Sectors identification

In [ ]:
import pandas as pd

# Load edge list
edges = pd.read_csv("bluesky_large_edges_unweighted_10000 (1).csv")

# Extract unique tickers
tickers = pd.unique(
    pd.concat([edges["source"], edges["target"]], ignore_index=True)
)

ticker_df = pd.DataFrame({
    "ticker": sorted(tickers)
})

print(f"Number of unique tickers: {len(ticker_df)}")
display(ticker_df.head())

In [ ]:
!pip install yfinance

In [ ]:
import yfinance as yf
import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

# Canonical normalization used in every notebook that consumes sector metadata.
MISSING_SECTOR_VALUES = {
    "", "unknown", "none", "n/a", "na", "null", "nan"
}

def normalize_sector(series):
    sector = series.astype("string").str.strip()
    invalid = sector.isna() | sector.str.casefold().isin(MISSING_SECTOR_VALUES)
    return sector.mask(invalid, pd.NA)

def get_metadata(ticker):
    try:
        info = yf.Ticker(ticker).info

        return pd.Series({
            "company": info.get("longName"),
            "sector": info.get("sector"),
            "industry": info.get("industry"),
            "country": info.get("country"),
            "exchange": info.get("exchange")
        })

    except Exception:
        return pd.Series({
            "company": None,
            "sector": None,
            "industry": None,
            "country": None,
            "exchange": None
        })

metadata = ticker_df["ticker"].progress_apply(get_metadata)

ticker_metadata = pd.concat([ticker_df, metadata], axis=1)

ticker_metadata["ticker"] = (
    ticker_metadata["ticker"].astype("string").str.strip().str.upper()
)
ticker_metadata["sector"] = normalize_sector(ticker_metadata["sector"])

In [ ]:
# Count known and missing sectors after canonical normalization.
missing = int(ticker_metadata["sector"].isna().sum())
known = int(ticker_metadata["sector"].notna().sum())

print(f"Tickers with known sector: {known}/{len(ticker_metadata)}")
print(f"Tickers without sector: {missing}/{len(ticker_metadata)}")
assert known + missing == len(ticker_metadata)

ticker_metadata[ticker_metadata["sector"].isna()].head(20)

In [ ]:
# salva il dataset in un file CSV
ticker_metadata.to_csv("ticker_metadata.csv", index=False)
print ("Ticker metadata saved to ticker_metadata.csv")

In [ ]:
import os

print(os.path.exists("ticker_metadata.csv"))

In [ ]:
import pandas as pd

df = pd.read_csv("ticker_metadata.csv")
df["ticker"] = df["ticker"].astype("string").str.strip().str.upper()
df["sector"] = normalize_sector(df["sector"])
df.head()

In [ ]:
print(df.shape)
print(df.columns)

In [ ]:
print(df.info())